In [1]:
import pandas as pd

In [3]:
# Load the dataset using pandas
df = pd.read_csv(
    'C:\\Users\\amand\\OneDrive\\Documents\\Electricity Bill and Consumption Analyzer\\raw data\\household_power_consumption.txt',
    sep=';',              # The file is semicolon-delimited
    low_memory=False,     # Prevents DtypeWarning for mixed types
    na_values=['?']       # Automatically treat '?' characters as NaN (missing values)
)

# Display the first 5 rows and basic info
print(df.head())
print(df.info())

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total

In [6]:
parse_dates={'datetime': ['Date', 'Time']} #combine Date and Time columns into a single datetime column
infer_datetime_format=True,
index_col='datetime' # Sets the new datetime as the index

Combining the Date and Time columns into a single timestamp is essential because it transforms two separate text strings into a true datetime object that Python and Pandas understand as continuous time.

In [7]:
df.head

<bound method NDFrame.head of                Date      Time  Global_active_power  Global_reactive_power  \
0        16/12/2006  17:24:00                4.216                  0.418   
1        16/12/2006  17:25:00                5.360                  0.436   
2        16/12/2006  17:26:00                5.374                  0.498   
3        16/12/2006  17:27:00                5.388                  0.502   
4        16/12/2006  17:28:00                3.666                  0.528   
...             ...       ...                  ...                    ...   
2075254  26/11/2010  20:58:00                0.946                  0.000   
2075255  26/11/2010  20:59:00                0.944                  0.000   
2075256  26/11/2010  21:00:00                0.938                  0.000   
2075257  26/11/2010  21:01:00                0.934                  0.000   
2075258  26/11/2010  21:02:00                0.932                  0.000   

         Voltage  Global_intensity  Sub_meter

In [8]:
df.dtypes

Date                      object
Time                      object
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

As seen above the power readings are integers, so i dont need to change any of the columns. If they were objects, i would not be able plot them or run mathematical operations on them. The next step is to remove negative values, missing columns, as well as invalid time stamps.

In [9]:
# Count missing values per column
missing_counts = df.isnull().sum()

# Calculate the percentage of missing values per column
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Combine them into a neat summary table
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentage
})

print(missing_summary)

                       Missing Count  Percentage (%)
Date                               0        0.000000
Time                               0        0.000000
Global_active_power            25979        1.251844
Global_reactive_power          25979        1.251844
Voltage                        25979        1.251844
Global_intensity               25979        1.251844
Sub_metering_1                 25979        1.251844
Sub_metering_2                 25979        1.251844
Sub_metering_3                 25979        1.251844


In [10]:
# If your datetime is set as the index:
invalid_timestamps = df.index.isna().sum()

print(f"Number of invalid timestamps: {invalid_timestamps}")

# To view the rows with invalid timestamps (if any):
print(df[df.index.isna()])

Number of invalid timestamps: 0
Empty DataFrame
Columns: [Date, Time, Global_active_power, Global_reactive_power, Voltage, Global_intensity, Sub_metering_1, Sub_metering_2, Sub_metering_3]
Index: []


Physically, power consumption, voltage, and electrical intensity measurements cannot be negative. If there are negative numbers, they are usually error codes or corrupt data points.

In [11]:
# List of numeric measurement columns
numeric_cols = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]

# Count how many negative values exist in each numeric column
negative_counts = (df[numeric_cols] < 0).sum()
print("Negative Readings Count:")
print(negative_counts)

Negative Readings Count:
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64


In [13]:
# 1. Create a temporary 'Hour' column from the datetime index
df['Hour'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
).dt.hour

# 2. Fill missing values using the median of that specific hour
# transform('median') calculates the median for each hour group and aligns it back to the original row shape
df[numeric_cols] = df[numeric_cols].fillna(
    df.groupby('Hour')[numeric_cols].transform('median')
)

# 3. Drop the temporary 'Hour' column
df = df.drop(columns=['Hour'])

# 4. Verify that all missing values are now gone
print("Remaining missing values:")
print(df.isnull().sum())

Remaining missing values:
Date                     0
Time                     0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64


In [14]:
# Save to a CSV file in your current project folder
df.to_csv('cleaned_household_power.csv')

print("Dataset successfully saved as 'cleaned_household_power.csv'!")

Dataset successfully saved as 'cleaned_household_power.csv'!
